# cudnn.benchmark 로 인한 학습 손실 흔들림

## 목적

`wip/tracker-tensorboard-wandb` 브랜치에서 텐서보드 기록을 붙인 뒤
학습 손실이 run 마다 미세하게 달라 보인 현상의 **원인을 특정**함.

가려낼 것 3가지:

1. tracker(`none` · `tensorboard` · `both`) 가 손실을 바꾸는가
2. `torch.backends.cudnn.benchmark = True` 가 바꾸는가
3. 여러 run 을 **동시에** 돌리는 경합이 영향을 주는가

## 배경 — 이전 세션이 남긴 부분 결과

커밋 `cbfce22` 의 메시지에 중단 지점이 적혀 있음.
`notebooks/_runs/cudnn_benchmark_jitter.csv` 에 16 run 중 **7 run** 만 남아 있고,
나머지는 dataloader worker 가 D 상태로 멈춰 측정을 못 끝냈음.

그 7 run 의 `ep1_train` 은 세 값으로 갈렸음 —
`31.535068893432616` · `31.538272094726562` · `31.544298553466795`.
`--tracker none` 만 쓴 run 들끼리도 세 값이 다 나왔음.

## 이번 조사에서 건드리지 않은 것

- `wesep/utils/utils.py` 의 `set_seed()` — 원본 그대로 둠.
  `cudnn.benchmark` 는 probe 안에서 `set_seed()` **뒤에** 덮어써서 조작함
- `confs/bsrnn.yaml` · `confs/debug.yaml` — 읽기만 함
- `wesep/bin/train.py` · `wesep/utils/executor.py` — 읽기만 함
- conda 환경 `wesep2` — 패키지를 설치하지 않음.
  `wespeaker` 는 스크래치패드에 클론해 `PYTHONPATH` 로만 얹음

## 막힌 것 — 원래 16 run 을 그대로 재현할 수 없음

| 항목 | 상태 |
|---|---|
| `examples/librimix/tse/v2/data/` | 없음 (gitignore 대상이라 컨테이너 재시작으로 소멸) |
| `/workspace/DB/Libri2Mix` | 없음 |
| `exp/stat_*` 이전 run | 0개 |

그래서 `run.sh --debug true` 의 stage 3 경로는 **실행 불가**임.
대신 데이터로더만 빼고 나머지는 실제 학습 경로를 그대로 쓰는 probe 를 씀 —
`notebooks/_bench/train_step_probe.py`.

| 요소 | 실제 학습과 같은가 |
|---|---|
| 모델 | 같음 — `confs/bsrnn.yaml` 의 BSRNN (ResNet34 화자 인코더 포함) |
| 손실 | 같음 — `SISDR` = `auraloss.time.SISDRLoss` (`wesep/utils/losses.py:24`) |
| 최적화 순서 | 같음 — `wesep/utils/executor.py:208-214` |
| 손실 집계 | **조사 당시 같았음** — `sum(losses)/len(losses)`.<br>그 뒤 본 학습만 샘플 수 가중 누적으로 바뀜(`wesep/utils/executor.py:180-184`). `drop_last: true` 라 배치 크기가 모두 같아 **평균값은 동일**하고(실측 차이 1.5e-08), 이 조사는 첫 두 스텝의 loss 를 비트 단위로 대조한 것이라 집계 방식과 무관함 |
| 입력 | **다름** — Libri2Mix 대신 고정 난수 배치.<br>전역 RNG 와 분리된 `torch.Generator(1234)` 로 만들어 run 사이에 동일 |

입력이 다르므로 **손실의 절대값은 실제 학습과 비교할 수 없음.**
이 조사가 보는 것은 오직 **같은 조건의 run 끼리 값이 일치하는가** 임.


## 환경

In [1]:
import sys
import torch

print("python  ", sys.version.split()[0])
print("torch   ", torch.__version__)
print("cudnn   ", torch.backends.cudnn.version())
print("cuda    ", torch.version.cuda)
print("gpu     ", torch.cuda.get_device_name(0), "x", torch.cuda.device_count())
print()
print("torch.backends.cuda.matmul.allow_tf32 :", torch.backends.cuda.matmul.allow_tf32)
print("torch.backends.cudnn.allow_tf32       :", torch.backends.cudnn.allow_tf32)
print("torch.get_float32_matmul_precision()  :", torch.get_float32_matmul_precision())

python   3.9.25
torch    2.7.1+cu128
cudnn    90701
cuda     12.8
gpu      NVIDIA GeForce RTX 3090 x 4

torch.backends.cuda.matmul.allow_tf32 : False
torch.backends.cudnn.allow_tf32       : True
torch.get_float32_matmul_precision()  : highest


## 실험 행렬

각 프로세스는 `CUDA_VISIBLE_DEVICES` 로 **GPU 1장만** 봄.

| 블록 | 축 | run 수 |
|---|---|---|
| `par4` — 4-way 병렬 | `cudnn_benchmark` {True,False} × `tracker` {none,tensorboard} × `gpu` {0,1,2,3} × `rep` {1,2} | 32 |
| `solo` — 단독 실행 | `cudnn_benchmark` {True,False} × `rep` {1..4}, GPU 0 한 장씩 순차 | 8 |

`solo` 블록을 넣은 이유 — `cudnn.benchmark` 는 후보 알고리즘을 **실행 시간으로 재서** 고르므로,
4개를 동시에 돌릴 때와 혼자 돌릴 때 선택이 달라질 수 있음.

실행한 명령:

```bash
export PYTHONPATH=<스크래치패드>/wespeaker_src
CUDA_VISIBLE_DEVICES=$gpu python notebooks/_bench/train_step_probe.py \
    --gpu $gpu --tracker $tracker --benchmark $bm --rep $rep \
    --tag $tag --epochs 3 --steps 5 --exp_dir <스크래치패드>/probe_exp/$name
```

`--epochs 3 --steps 5` 는 `confs/debug.yaml` 의 `num_epochs: 3` · `steps_per_epoch: 5` 와 같은 크기임.


## 원자료에 손을 댄 곳 — 먼저 밝힘

`_runs/train_step_probe.csv` 는 **자동 생성물이 아니라 두 번 손을 댄 파일**임.
근거로 쓰는 표이므로 무엇을 했는지 먼저 적음.

| 시점 | 무엇을 | 왜 | 되돌릴 사본 |
|---|---|---|---|
| 측정 중 | `par4 / tensorboard / cudnn_benchmark=False / rep=1` **4행에서 여분 필드 1개 제거** | 측정이 도는 중에 `LEFT` 에 열을 추가했다 되돌렸고,<br>그 사이에 시작된 4 run 이 헤더(15열)보다 1열 많은 16열로 기록됨.<br>여분 값은 `index 3` 의 `cudnn_deterministic='False'` 였고 그 자리만 뺌 | 스크래치패드의<br>`train_step_probe.csv.bak` |
| 측정 후 | `cudnn_deterministic` · `use_det_algos` **두 열을 추가하고 41행 전부 `False` 로 채움** | 조건이 CSV 에 안 남아 있던 것을 메움.<br>`run_matrix.sh` 가 두 인자를 한 번도 넘기지 않았고<br>`run_blockC.sh` 는 실행 전 취소됐으며(`blockC.log` 0바이트),<br>`set_seed` 도 `deterministic` 을 건드리지 않으므로 **전부 torch 기본값 `False`** 임 | 스크래치패드의<br>`train_step_probe.csv.before_backfill` |

**삭제한 측정값은 없음.** 첫 번째는 열 정렬을 맞춘 것이고 두 번째는 빈 열을 채운 것임.

이 사고 때문에 한 번 **잘못된 중간 보고**를 했음 — 열이 밀린 상태로 집계해
`cudnn_benchmark=False · tracker=tensorboard` 의 `step1` 갈래를 5 로 읽었으나,
복구 후 실제 값은 **1** 이었음.


## 측정 원자료 적재

In [2]:
import pandas as pd
from pathlib import Path

CSV = Path("_runs/train_step_probe.csv")
df = pd.read_csv(CSV, dtype=str)
df = df[df["tag"] != "smoke"].copy()
print(f"{len(df)} run")
display(df[["tag", "tracker", "cudnn_benchmark", "gpu", "rep",
            "step1_loss", "ep1_train", "ep3_train"]])

40 run


,tag,tracker,cudnn_benchmark,gpu,rep,step1_loss,ep1_train,ep3_train
1,par4,none,True,2,1,49.332359313964844,50.018273162841794,53.89347915649414
2,par4,none,True,1,1,49.332359313964844,50.025943756103516,54.053443908691406
3,par4,none,True,0,1,49.332359313964844,50.024871063232425,53.84082717895508
4,par4,none,True,3,1,49.33146667480469,50.035448455810545,49.888404083251956
5,par4,none,True,3,2,49.332359313964844,50.01483840942383,53.402208709716795
6,par4,none,True,1,2,49.332359313964844,50.02437210083008,54.091252136230466
7,par4,none,True,0,2,49.332359313964844,50.01335144042969,51.35525665283203
8,par4,none,True,2,2,49.330509185791016,50.074365234375,52.36279830932617
9,par4,tensorboard,True,3,1,49.33146667480469,50.03590469360351,53.85116195678711
10,par4,tensorboard,True,2,1,49.333248138427734,50.031062316894534,51.543172454833986


## 흔들림의 정의

같은 조건 안에서 값이 **몇 갈래로 갈리는지** 를 셈.
`1` 이면 그 조건의 모든 run 이 완전히 일치한 것임.

부동소수점이라 `repr(float)` 문자열을 그대로 비교함 — 반올림으로 뭉개지 않게.

| 열 | 뜻 |
|---|---|
| `n_distinct_step1` | 첫 스텝 손실의 갈래 수. **순수 forward 차이** (가중치 갱신 전) |
| `n_distinct_ep1` | 1 에포크 평균 손실의 갈래 수 |
| `n_distinct_ep3` | 3 에포크 평균 손실의 갈래 수 |
| `n_distinct_weight_sum` | 학습이 끝난 뒤 전체 파라미터 합의 갈래 수 |


In [3]:
def spread(df, keys):
    g = df.groupby(keys)
    out = g.agg(
        n_run=("tag", "size"),
        n_distinct_step1=("step1_loss", "nunique"),
        n_distinct_ep1=("ep1_train", "nunique"),
        n_distinct_ep3=("ep3_train", "nunique"),
        n_distinct_weight_sum=("weight_sum_after", "nunique"),
    ).reset_index()
    return out


display(spread(df, ["tag", "cudnn_benchmark"]))

,tag,cudnn_benchmark,n_run,n_distinct_step1,n_distinct_ep1,n_distinct_ep3,n_distinct_weight_sum
0,par4,False,16,1,16,16,16
1,par4,True,16,5,16,16,16
2,solo,False,4,1,4,4,4
3,solo,True,4,2,4,4,4


### tracker 별로 갈라 봄 — tracker 가 원인이라면 여기서 갈려야 함

In [4]:
display(spread(df, ["tag", "cudnn_benchmark", "tracker"]))

,tag,cudnn_benchmark,tracker,n_run,n_distinct_step1,n_distinct_ep1,n_distinct_ep3,n_distinct_weight_sum
0,par4,False,none,8,1,8,8,8
1,par4,False,tensorboard,8,1,8,8,8
2,par4,True,none,8,3,8,8,8
3,par4,True,tensorboard,8,4,8,8,8
4,solo,False,none,4,1,4,4,4
5,solo,True,none,4,2,4,4,4


### GPU 별 — 같은 GPU 안의 반복은 일치하는가

In [5]:
display(spread(df, ["tag", "cudnn_benchmark", "gpu"]))

,tag,cudnn_benchmark,gpu,n_run,n_distinct_step1,n_distinct_ep1,n_distinct_ep3,n_distinct_weight_sum
0,par4,False,0,4,1,4,4,4
1,par4,False,1,4,1,4,4,4
2,par4,False,2,4,1,4,4,4
3,par4,False,3,4,1,4,4,4
4,par4,True,0,4,1,4,4,4
5,par4,True,1,4,3,4,4,4
6,par4,True,2,4,3,4,4,4
7,par4,True,3,4,2,4,4,4
8,solo,False,0,4,1,4,4,4
9,solo,True,0,4,2,4,4,4


### ep1_train 실제 값 — 몇 갈래인지 눈으로 확인

In [6]:
for (tag, bm), sub in df.groupby(["tag", "cudnn_benchmark"]):
    vals = sorted(sub["ep1_train"].unique())
    print(f"[tag={tag} cudnn_benchmark={bm}]  run {len(sub)}개 -> 갈래 {len(vals)}개")
    for v in vals:
        who = sub[sub["ep1_train"] == v]
        tags = ", ".join(f"g{r.gpu}/{r.tracker}/r{r.rep}" for r in who.itertuples())
        print(f"    {v:<24} <- {tags}")
    print()

[tag=par4 cudnn_benchmark=False]  run 16개 -> 갈래 16개
    50.15786285400391        <- g2/none/r1
    50.15797576904297        <- g0/none/r2
    50.15811462402344        <- g3/tensorboard/r2
    50.15821914672851        <- g3/none/r2
    50.159158325195314       <- g1/none/r2
    50.15955581665039        <- g0/tensorboard/r2
    50.159699249267575       <- g2/tensorboard/r1
    50.16105270385742        <- g3/none/r1
    50.16255493164063        <- g0/none/r1
    50.16384506225586        <- g1/tensorboard/r2
    50.164990234375          <- g0/tensorboard/r1
    50.16960067749024        <- g2/tensorboard/r2
    50.17024307250976        <- g3/tensorboard/r1
    50.176849365234375       <- g1/none/r1
    50.17878875732422        <- g2/none/r2
    50.186292266845705       <- g1/tensorboard/r1

[tag=par4 cudnn_benchmark=True]  run 16개 -> 갈래 16개
    50.01335144042969        <- g0/none/r2
    50.01483840942383        <- g3/none/r2
    50.01599044799805        <- g0/tensorboard/r2
    50.018273162

### 스텝별 손실 — 어느 스텝부터 갈라지는가

In [7]:
rows = []
for r in df.itertuples():
    steps = r.all_steps.split()
    rows.append({"tag": r.tag, "bm": r.cudnn_benchmark, "tracker": r.tracker,
                 "gpu": r.gpu, "rep": r.rep,
                 **{f"s{i+1}": steps[i] for i in range(min(5, len(steps)))}})
steps_df = pd.DataFrame(rows).sort_values(["tag", "bm", "tracker", "rep", "gpu"])
display(steps_df)

,tag,bm,tracker,gpu,rep,s1,s2,s3,s4,s5
19,par4,False,none,0,1,49.32976150512695,51.86861801147461,48.127838134765625,54.186824798583984,47.29973220825195
17,par4,False,none,1,1,49.32976150512695,51.856056213378906,48.10343933105469,54.2072639465332,47.387725830078125
16,par4,False,none,2,1,49.32976150512695,51.87156677246094,48.1087646484375,54.181209564208984,47.298011779785156
18,par4,False,none,3,1,49.32976150512695,51.875301361083984,48.10247039794922,54.19613265991211,47.301597595214844
21,par4,False,none,0,2,49.32976150512695,51.867637634277344,48.10934066772461,54.17407989501953,47.309059143066406
22,par4,False,none,1,2,49.32976150512695,51.863563537597656,48.11660385131836,54.18330383300781,47.30255889892578
20,par4,False,none,2,2,49.32976150512695,51.862579345703125,48.099327087402344,54.20635986328125,47.39591598510742
23,par4,False,none,3,2,49.32976150512695,51.861602783203125,48.10398864746094,54.18537902832031,47.31036376953125
26,par4,False,tensorboard,0,1,49.32976150512695,51.86061477661133,48.111351013183594,54.20603942871094,47.31718444824219
25,par4,False,tensorboard,1,1,49.32976150512695,51.864585876464844,48.087860107421875,54.207862854003906,47.44139099121094


---

# 2차 조사 — 요인 스윕 (`factor_screen.py`)

1차(`train_step_probe.py`)는 3 에포크 × 5 스텝 = 15 스텝을 돌아 run 당 40~60초였음.
**사용자 제안**으로 설계를 바꿈 — 컴파일 미사용, 배치 최소한만 돌고,
의심 요인을 CSV 의 열로 두고 하나씩 바꿔 가며 손실을 추적함.
run 당 **약 2.7초**로 줄었음.

## 1 배치가 아니라 2 스텝을 도는 이유

1차 결과가 근거임 — `cudnn.benchmark=False` 인 16 run 의 **첫 배치 손실은 한 값으로 완전히 일치**했지만
에포크 손실은 16갈래로 갈렸음. 즉 첫 배치만 재면 역전파 쪽 원인이 **보이지 않음**.

그래서 2 스텝을 돌되, 그 사이에서 기울기를 직접 집어냄:

| 측정값 | 재는 지점 | 이것만 달라지면 |
|---|---|---|
| `loss_step1` | 순전파 직후 (= "첫 배치 손실") | **순전파**가 원인 — 알고리즘 선택 |
| `grad_sum_step1` | `backward()` 후 `optimizer.step()` **전** | **역전파**가 원인 — 누적 순서 |
| `loss_step2` | 갱신 뒤 두 번째 배치의 순전파 | 둘이 합쳐진 결과 |

## 요인 — CSV 왼쪽 열

| 요인 | wesep 원본 상태 | 의심하는 이유 |
|---|---|---|
| `cudnn_benchmark` | **True 로 강제** (`wesep/utils/utils.py:116`) | 알고리즘을 실행 시간으로 골라 run 마다 달라질 수 있음 |
| `cudnn_deterministic` | **주석 처리됨** (`wesep/utils/utils.py:115`) | 고른 알고리즘이 atomicAdd 를 쓰면 누적 순서가 매번 다름 |
| `cudnn_allow_tf32` | True (PyTorch 기본값) | Ampere 에서 conv 가 TF32 로 돌아 알고리즘 차이가 크게 남음 |
| `matmul_allow_tf32` | False (PyTorch 기본값) | 켜면 matmul 정밀도가 바뀜 |
| `use_det_algos` | False | cuDNN 밖(scatter_add 등)까지 결정적 구현 강제. **문서 목록에 LSTM·GRU 는 없음** |
| `cudnn_enabled` | True | 끄면 cuDNN 을 아예 안 씀 — 원인이 cuDNN 안인지 밖인지 가름 |
| `amp` | `enable_amp: false` (`confs/bsrnn.yaml:31`) | 켜지면 GradScaler 가 스텝을 건너뛸 수 있음 |
| `enr_len` | 실제 학습은 **가변** | 등록 발화가 통째로 읽히고 collate 가 배치 최소 길이로 자름(`processor.py:477-479`).<br>ResNet34 의 conv 입력 `t` 축이 스텝마다 바뀌면 benchmark 가 **매 스텝 재탐색**함 |
| `tracker` | — | 사용자가 처음 의심한 요인. 대조군으로 유지 |


## 요인 스윕 원자료

In [8]:
fs = pd.read_csv("_runs/factor_screen.csv", dtype=str)
fs = fs[~fs["arm"].str.startswith("smoke")].copy()
print(f"{len(fs)} run")
display(fs[["arm", "cudnn_benchmark", "cudnn_deterministic", "cudnn_allow_tf32",
            "matmul_allow_tf32", "use_det_algos", "cudnn_enabled", "amp",
            "enr_len", "tracker", "gpu", "rep",
            "loss_step1", "grad_sum_step1", "loss_step2"]])

96 run


,arm,cudnn_benchmark,cudnn_deterministic,cudnn_allow_tf32,matmul_allow_tf32,use_det_algos,cudnn_enabled,amp,enr_len,tracker,gpu,rep,loss_step1,grad_sum_step1,loss_step2
1,base,True,False,True,False,False,True,False,335,none,1,1,49.333248138427734,-14115.563940223818,51.78778839111328
2,base,True,False,True,False,False,True,False,335,none,3,1,49.332359313964844,-14106.027730963495,51.78809356689453
3,base,True,False,True,False,False,True,False,335,none,0,1,49.331382751464844,-14096.283139151936,51.80695343017578
4,base,True,False,True,False,False,True,False,335,none,2,1,49.332359313964844,-14107.16338041216,51.792415618896484
5,base,True,False,True,False,False,True,False,335,none,2,2,49.332359313964844,-14107.177522596736,51.801292419433594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,bm_off__det_on__det_algos,False,True,True,False,True,True,False,335,none,3,1,49.32976150512695,-13749.078503751263,51.84455490112305
93,bm_off__det_on__det_algos,False,True,True,False,True,True,False,335,none,1,2,49.32976150512695,-13749.078503751263,51.84455490112305
94,bm_off__det_on__det_algos,False,True,True,False,True,True,False,335,none,0,2,49.32976150512695,-13749.078503751263,51.84455490112305
95,bm_off__det_on__det_algos,False,True,True,False,True,True,False,335,none,3,2,49.32976150512695,-13749.078503751263,51.84455490112305


## arm 별 갈래 수 — **갈래 1 이면 그 요인이 흔들림을 잡은 것**

`갈래` 는 같은 arm 의 8 run(GPU 4 × rep 2) 안에서 값이 몇 가지로 나뉘는지임.
`base대비` 는 중앙값의 이동량 — 흔들림과는 **별개**의 이야기임.
값이 이동해도 흔들림은 남을 수 있고, 그 반대도 있음.


In [9]:
ARM_ORDER = ["base", "bm_off", "det_on", "cudnn_tf32_off", "matmul_tf32_on",
             "det_algos_on", "cudnn_off", "amp_on", "tracker_tb", "enr_vary",
             "bm_off__det_on", "bm_off__det_algos",
             "bm_off__det_on__det_algos", "cudnn_off__det_algos"]
M = ["loss_step1", "grad_sum_step1", "loss_step2"]

for c in M:
    fs[c + "_f"] = fs[c].astype(float)

base_med = {c: fs.loc[fs.arm == "base", c + "_f"].median() for c in M}

g = fs.groupby("arm")
tbl = pd.DataFrame({
    "n": g.size(),
    **{f"{c}_갈래": g[c].nunique() for c in M},
    **{f"{c}_base대비": (g[c + "_f"].median() - base_med[c]).map("{:+.6g}".format)
       for c in M},
    "sec": g["sec"].apply(lambda s: round(s.astype(float).mean(), 1)),
})
tbl = tbl.reindex([a for a in ARM_ORDER if a in tbl.index])
display(tbl)

,n,loss_step1_갈래,grad_sum_step1_갈래,loss_step2_갈래,loss_step1_base대비,grad_sum_step1_base대비,loss_step2_base대비,sec
arm,,,,,,,,
base,8,3,8,8,+0,+0,+0,2.7
bm_off,8,1,8,8,-0.00259781,+349.311,+0.0661926,2.3
det_on,8,3,4,4,+0,+0.953176,-0.00743103,3.3
cudnn_tf32_off,8,1,8,8,-0.00673294,-576.01,+0.0427761,3.0
matmul_tf32_on,8,2,8,8,-0.00761414,+34.274,+0.00158119,2.9
det_algos_on,8,4,6,6,+0,+0.88879,-0.00743103,3.3
amp_on,8,1,0,1,-0.000804901,+nan,+3.51701,2.6
tracker_tb,8,3,8,8,+0,+0.0240689,-0.0042305,2.7
enr_vary,8,3,8,8,+0,+0.0143186,-2.14592,2.8


### 갈래 1 을 그대로 믿으면 안 됨 — `nan` 검사를 먼저

`amp_on` 이 실제로 걸린 함정임. 세 측정값이 모두 갈래 1 로 나왔지만
`grad_sum_step1` 이 8 run 전부 **`nan`** 이라 같아진 것이었음.

AMP 를 켜면 `GradScaler` 의 초기 스케일(65536)에 fp16 기울기가 넘쳐 `inf` 가 되고,
`scaler.step()` 이 **옵티마이저 스텝을 통째로 건너뜀** — 가중치가 안 바뀌니
`loss_step2` 도 전부 같아짐. 원인을 잡은 것이 아니라 **학습이 안 일어난 것**임.


In [10]:
import numpy as np

valid = fs.groupby("arm").apply(
    lambda d: bool(np.isfinite(d[[c + "_f" for c in M]].to_numpy()).all()))
tbl.insert(0, "유효", valid.reindex(tbl.index).map({True: "O", False: "X(nan/inf)"}))
display(tbl)

,유효,n,loss_step1_갈래,grad_sum_step1_갈래,loss_step2_갈래,loss_step1_base대비,grad_sum_step1_base대비,loss_step2_base대비,sec
arm,,,,,,,,,
base,O,8,3,8,8,+0,+0,+0,2.7
bm_off,O,8,1,8,8,-0.00259781,+349.311,+0.0661926,2.3
det_on,O,8,3,4,4,+0,+0.953176,-0.00743103,3.3
cudnn_tf32_off,O,8,1,8,8,-0.00673294,-576.01,+0.0427761,3.0
matmul_tf32_on,O,8,2,8,8,-0.00761414,+34.274,+0.00158119,2.9
det_algos_on,O,8,4,6,6,+0,+0.88879,-0.00743103,3.3
amp_on,X(nan/inf),8,1,0,1,-0.000804901,+nan,+3.51701,2.6
tracker_tb,O,8,3,8,8,+0,+0.0240689,-0.0042305,2.7
enr_vary,O,8,3,8,8,+0,+0.0143186,-2.14592,2.8


### 흔들림을 잡은 arm 만 추림 (유효한 것만)

In [11]:
fixed = tbl[(tbl["유효"] == "O")
            & (tbl["loss_step1_갈래"] == 1)
            & (tbl["grad_sum_step1_갈래"] == 1)
            & (tbl["loss_step2_갈래"] == 1)]
if len(fixed):
    print("세 측정값이 모두 한 값으로 모인 arm (nan 없이):")
    display(fixed)
    for arm in fixed.index:
        sub = fs[fs.arm == arm]
        print(f"\n[{arm}] 8 run 의 실제 값")
        for r in sub.sort_values(["rep", "gpu"]).itertuples():
            print(f"  g{r.gpu} r{r.rep}  loss1={r.loss_step1:<20} "
                  f"grad={r.grad_sum_step1:<22} loss2={r.loss_step2}")
else:
    print("유효하면서 세 측정값이 모두 모인 arm 이 없음")

세 측정값이 모두 한 값으로 모인 arm (nan 없이):


,유효,n,loss_step1_갈래,grad_sum_step1_갈래,loss_step2_갈래,loss_step1_base대비,grad_sum_step1_base대비,loss_step2_base대비,sec
arm,,,,,,,,,
bm_off__det_on,O,8,1,1,1,-0.00259781,+358.099,+0.0437965,2.2
bm_off__det_algos,O,8,1,1,1,-0.00259781,+358.099,+0.0437965,2.7
bm_off__det_on__det_algos,O,8,1,1,1,-0.00259781,+358.099,+0.0437965,2.4



[bm_off__det_on] 8 run 의 실제 값
  g0 r1  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g1 r1  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g2 r1  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g3 r1  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g0 r2  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g1 r2  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g2 r2  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g3 r2  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305

[bm_off__det_algos] 8 run 의 실제 값
  g0 r1  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g1 r1  loss1=49.32976150512695    grad=-13749.078503751263    loss2=51.84455490112305
  g2 r1  loss1=49.32976150512695    grad=-13749.0785037

---

# 결론

## 원인은 **둘**이고, 둘 다 꺼야 멈춤

| 층 | 원인 | 고치는 스위치 | 단독으로 충분한가 |
|---|---|---|---|
| 1층 · 순전파 | cuDNN 이 conv 알고리즘을 **실행 시간으로 재서** 고름.<br>측정이 프로세스마다 달라 다른 알고리즘이 뽑힘 | `wesep/utils/utils.py:116`<br>`cudnn.benchmark = True` → `False` | ❌ 역전파가 남음 (`bm_off` = 1/8/8) |
| 2층 · 역전파 | 고른 알고리즘 자체가 **비결정적** — 누적 순서가 매번 다름 | `wesep/utils/utils.py:115` 의<br>**주석 처리된** `cudnn.deterministic = True` 를 살림 | ❌ 순전파가 남음 (`det_on` = 3/4/4) |

**둘을 같이 적용한 `bm_off__det_on` 에서만 8 run 이 비트 단위로 일치함** (1/1/1).

wesep 원본이 111줄을 주석 처리하고 112줄만 켜 둔 것이
**정확히 흔들림을 만드는 조합**임.

## `torch.use_deterministic_algorithms` 는 불필요

`bm_off__det_on` · `bm_off__det_algos` · `bm_off__det_on__det_algos` 세 arm 의
값이 **서로도 완전히 같음.** 즉 여기서 `use_deterministic_algorithms(True)` 는
`cudnn.deterministic = True` 와 같은 일만 함.

`det_algos_on` 단독이 갈래 4/6/6 이었던 이유도 같음 —
`use_deterministic_algorithms(True)` 는 후보를 결정적 알고리즘으로 좁힐 뿐
**`benchmark` 를 끄지 않아** 그 후보들 사이에서 시간 재기 추첨이 계속됨.

## 텐서보드는 원인이 아님

`tracker_tb` 가 `base` 와 **모든 칸에서 동일**함 — 갈래 3/8/8,
`loss_step1` 중앙값 이동 `+0`.

흔들림은 텐서보드를 붙이기 **전부터 있었고**,
텐서보드가 학습 곡선을 그려주면서 **비로소 눈에 보이게 된 것**임.

## 남은 한계

| 항목 | 기존 문제점 | 해결방안 | 상태 |
|---|---|---|---|
| 실제 학습 경로 미확인 | Libri2Mix 가 사라져 `run.sh --debug true` 를 못 돌림.<br>이 조사는 데이터로더를 뺀 probe 임 | 합성 shard 로 stage 3 을 돌리는 것이 **가능**함<br>(목록·JSON 4개 + shard tar 2개 + 등록용 wav 수십 개) | **미실행** |
| `cudnn_off` arm 실패 | `cudnn.enabled=False` 면 `nn.LSTM` 이 비 fused 폴백으로 떨어져<br>RTX 3090 24 GB 에서 OOM (`bsrnn.py:41`) | 배치를 줄여 재시도.<br>다만 `bm_off__det_on` 이 1/1/1 을 냈으므로<br>**원인이 cuDNN 안이라는 답은 이미 나옴** | 불필요해짐 |
| 등록 발화 가변 길이 | 실제 학습은 배치마다 `t` 축이 달라 benchmark 가 매 스텝 재탐색함.<br>probe 는 2 스텝뿐이라 이 효과를 거의 못 담음 | 실제 파이프라인 확인 때 자연히 포함됨 | `enr_vary` arm 으로 부분 확인 (차이 없음) |
| 속도 대가 미측정 | `benchmark=False` 는 알고리즘 자동 선택을 포기하는 것이라<br>학습이 느려질 수 있음 | 같은 probe 로 `sec` 열 비교 가능하나<br>2 스텝은 너무 짧아 의미 없음 | **미측정** |
